In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# [GPT Link](https://chatgpt.com/share/69b327a6-da94-8005-80c3-f7f09081bc40)

## [Word File](https://docs.google.com/document/d/1ufwWzABa8MhAhmbVzq7El8mLDy3Ra8m-l1jWfWW9ZRE/edit?usp=sharing)

# [Policy_en CSV Link](https://drive.google.com/file/d/17ShQTtD0Ryy6wycrrLfkJ3Ak-80nkp-v/view?usp=sharing)

In [ ]:
#

In [ ]:
import pandas as pd
Policy_Csv = "/content/drive/MyDrive/datasets/Policy_en.csv"
df = pd.read_csv(Policy_Csv)
df

,policy_id,section,policy_name,rule_id,rule_description,threshold_value,condition,approval_required,risk_level
0,1,Financial,Hardware Purchase Policy,1,Purchases up to $1500 are auto-approved,1500.0,amount <= 1500,No,Medium
1,1,Financial,Hardware Purchase Policy,2,Purchases between $1501–$3000 require manager ...,3000.0,1500 < amount <= 3000,Manager,Medium
2,1,Financial,Hardware Purchase Policy,3,"Purchases above $3000 require manager, finance...",3000.0,amount > 3000,Manager+Finance+CTO,Medium
3,1,Financial,Hardware Purchase Policy,4,Annual hardware budget per employee is $2500,2500.0,annual_budget_limit,No,Medium
4,1,Financial,Hardware Purchase Policy,5,Vendor must be selected from approved vendor list,NaN,vendor_approved == true,No,Medium
5,2,Financial,Travel Expense Policy,1,Economy class required for flights under 6 hours,6.0,flight_hours <= 6,No,Medium
6,2,Financial,Travel Expense Policy,2,Business class allowed for flights over 6 hour...,6.0,flight_hours > 6 AND role == executive,Manager,Medium
7,2,Financial,Travel Expense Policy,3,Hotel cost must not exceed $250 regional,250.0,region == regional,No,Medium
8,2,Financial,Travel Expense Policy,4,Hotel cost must not exceed $400 international ...,400.0,region == international,No,Medium
9,2,Financial,Travel Expense Policy,5,Receipts mandatory for reimbursement,NaN,receipt_provided == true,No,Medium


In [ ]:
df= df.drop(columns=["policy_id","rule_id"])
df["full_text"] = df.astype(str).agg(" | ".join, axis=1)
df.head()

,section,policy_name,rule_description,threshold_value,condition,approval_required,risk_level,full_text
0,Financial,Hardware Purchase Policy,Purchases up to $1500 are auto-approved,1500.0,amount <= 1500,No,Medium,Financial | Hardware Purchase Policy | Purchas...
1,Financial,Hardware Purchase Policy,Purchases between $1501–$3000 require manager ...,3000.0,1500 < amount <= 3000,Manager,Medium,Financial | Hardware Purchase Policy | Purchas...
2,Financial,Hardware Purchase Policy,"Purchases above $3000 require manager, finance...",3000.0,amount > 3000,Manager+Finance+CTO,Medium,Financial | Hardware Purchase Policy | Purchas...
3,Financial,Hardware Purchase Policy,Annual hardware budget per employee is $2500,2500.0,annual_budget_limit,No,Medium,Financial | Hardware Purchase Policy | Annual ...
4,Financial,Hardware Purchase Policy,Vendor must be selected from approved vendor list,NaN,vendor_approved == true,No,Medium,Financial | Hardware Purchase Policy | Vendor ...


In [ ]:
df["full_text"][0]

'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'

In [ ]:
# Target
#1) Matching row with userQ  ( Semantic Search)
#2) Matching rule with condition col. (keyword Search)

In [ ]:
#  Semantic Search

# 1) Embedding


from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
policy_texts = df["full_text"].tolist()

policy_embeddings = embedder.encode(
    policy_texts,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
policy_texts[0]

'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'

In [ ]:
policy_embeddings[0]

array([-1.37061914e-02,  4.47236598e-02, -1.61637012e-02, -5.36067747e-02,
       -1.91182469e-03,  2.49939617e-02,  2.33572293e-02,  4.44731191e-02,
       -4.84552272e-02,  2.69257501e-02,  6.35468215e-02, -1.86100081e-02,
        6.56950194e-03, -5.27026579e-02, -8.51291865e-02, -1.72419809e-02,
        5.09684980e-02, -1.25831291e-01, -2.87466925e-02,  4.32613445e-03,
        2.74031181e-02, -3.38292122e-02, -1.66924689e-02,  1.64940860e-02,
        3.45937572e-02,  6.16535405e-03,  5.32277450e-02,  2.17648651e-02,
       -7.00197648e-03,  3.58746126e-02, -1.00474589e-01, -3.03701963e-02,
        9.29182172e-02, -5.24217673e-02,  6.00386821e-02, -7.98505619e-02,
       -6.13256879e-02, -1.00962795e-01, -3.90885957e-02, -1.18482850e-01,
        1.34699391e-02, -8.85257944e-02, -8.76408815e-02,  3.28780189e-02,
        6.45360574e-02,  2.00531203e-02, -6.23312108e-02,  6.48792461e-02,
        2.42649857e-03,  8.24130699e-02,  4.28119069e-03,  7.37885088e-02,
        4.21453267e-02, -

In [ ]:
userQ =  "i need left form my job in 3 Days"
userQ = "I need buy device with 4500$"

userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
)

In [ ]:
# userQ_embedding  and policy_embeddings

In [ ]:
score = (  userQ_embedding @ policy_embeddings.T).squeeze()

# Cosine Similarity apply with normalize_embeddings=True in embedder.encode()
# score = (userQ_embedding @ policy_embeddings.T).squeeze() / (
  #  np.linalg.norm(userQ_embedding) *
 #   np.linalg.norm(policy_embeddings, axis=1)
# )

top_k = 5
top_idx = np.argsort(-score)[:top_k]

for i in top_idx:
    print("Score:", round(score[i], 3))
    print("Text :", df.iloc[i]["full_text"])
    print("-"*60)

Score: 0.478
Text : Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
------------------------------------------------------------
Score: 0.369
Text : Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
------------------------------------------------------------
Score: 0.357
Text : Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
------------------------------------------------------------
Score: 0.311
Text : Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
------------------------------------------------------------
Score: 0.263
Text : Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list 

In [ ]:
#

In [ ]:
def retrive_cand(userQ,policy_embeddings):
  out = []
  userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
  )
  score = (  userQ_embedding @ policy_embeddings.T).squeeze()
  top_k = 5
  top_idx = np.argsort(-score)[:top_k]
  for i in top_idx:
    #  print("Score:", round(score[i], 3))

    out.append({
            "df_idx": int(i),
            "Score": round(float(score[i]), 3),
            "full_text": df.iloc[i]["full_text"],
            "condition": str(df.iloc[i]["condition"]),
        })

    # out.append({ "Score":round(score[i], 3)  ,  "full_text":df.iloc[i]["full_text"]})
     # print("Text :", df.iloc[i]["full_text"])
    #  print("-"*60)
  return out

# 2) Matching rule with condition col. (keyword Search)

In [ ]:
userQ =  "i need left form my job in 3 Days"
retrive_cand(userQ,policy_embeddings)

[{'df_idx': 16,
  'Score': 0.241,
  'full_text': 'Legal | Contract Termination Policy | Contracts must include minimum 30-day termination notice | 30.0 | termination_notice_days >= 30 | Legal | High',
  'condition': 'termination_notice_days >= 30'},
 {'df_idx': 25,
  'Score': 0.239,
  'full_text': 'HR | Remote Work Equipment Policy | Replacement allowed every 3 years | 3.0 | years_since_last_replacement >= 3 | Manager | Medium',
  'condition': 'years_since_last_replacement >= 3'},
 {'df_idx': 18,
  'Score': 0.161,
  'full_text': 'Legal | Contract Termination Policy | Indemnification clauses must be reviewed by legal | nan | indemnification_clause == true | Legal | High',
  'condition': 'indemnification_clause == true'},
 {'df_idx': 21,
  'Score': 0.158,
  'full_text': 'Legal | Vendor Agreement Policy | Payment terms must not exceed Net 60 unless CFO approval | 60.0 | payment_terms_days <= 60 | CFO_if_exceeds | High',
  'condition': 'payment_terms_days <= 60'},
 {'df_idx': 17,
  'Score'

In [ ]:
userQ = "I need buy device with 4500$"
top5Cand = retrive_cand(userQ,policy_embeddings)
top5Cand

[{'df_idx': 1,
  'Score': 0.478,
  'full_text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium',
  'condition': '1500 < amount <= 3000'},
 {'df_idx': 0,
  'Score': 0.369,
  'full_text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium',
  'condition': 'amount <= 1500'},
 {'df_idx': 2,
  'Score': 0.357,
  'full_text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium',
  'condition': 'amount > 3000'},
 {'df_idx': 3,
  'Score': 0.311,
  'full_text': 'Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium',
  'condition': 'annual_budget_limit'},
 {'df_idx': 4,
  'Score': 0.263,
  'full_text': 'Financial | Hardware Purchase Pol

In [ ]:
# 2) Matching rule with condition col. (keyword Search)

# A) extract number form userQ
# B) extract cond from full text in top5 cand
# C) check cond ???

In [ ]:
import re

def extract_amount(text):
    match = re.search(r'[\d,]+', text)
    if match:
        return int(match.group().replace(",", ""))
    return None

amount_userQ = extract_amount(userQ)
amount_userQ

4500

In [ ]:
def fullsystem(userQ,policy_embeddings):
  out = []
  amount_userQ = extract_amount(userQ) # 1

  userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
  )
  score = (  userQ_embedding @ policy_embeddings.T).squeeze()
  top_k = 5
  top_idx = np.argsort(-score)[:top_k]
  num = 0
  matched = []
  for i in top_idx:
   # print(f"----------- top {num+1} -----------")
    condition = df.iloc[i]["condition"]
    #print(condition)
    expr= condition.replace("amount",str(amount_userQ))
    #print(expr)
    try:
      if eval(expr) == True:
        matched.append({ "Score":round(score[i], 3)  ,  "full_text":df.iloc[i]["full_text"]})

    except:
      pass
    num +=1

  return matched


In [ ]:
userQ = "I need buy device with 4500$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.357),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium'}]

In [ ]:
userQ = "I need buy device with 1000$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.34),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'}]

In [ ]:
userQ = "I need buy device with 2500$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.437),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium'}]

# 3) Hybrid Retrieval (Semantic + BM25)

In [ ]:
!pip install rank-bm25

In [ ]:
# ---------------------------------
# 3) Hybrid Retrieval (Semantic + BM25)
# ---------------------------------

# install library (run once)
# !pip install rank-bm25

from rank_bm25 import BM25Okapi
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def simple_tokenize(text):

    # lower
    text = text.lower()

    # keep numbers
    tokens = re.findall(r"[a-zA-Z0-9]+", text)

    # remove stopwords
    tokens = [t for t in tokens if t not in stop_words]

    # stemming
    tokens = [stemmer.stem(t) for t in tokens]

    return tokens
# build BM25 index from policy texts
tokenized_corpus = [simple_tokenize(text) for text in policy_texts]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index ready")

BM25 index ready


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
def retrieve_bm25(userQ, policy_texts, bm25, top_k=5):

    query_tokens = simple_tokenize(userQ)

    scores = bm25.get_scores(query_tokens)

    top_idx = np.argsort(-scores)[:top_k]

    results = []

    for i in top_idx:

        results.append({
            "Score": round(scores[i],3),
            "full_text": policy_texts[i],
            "method":"bm25"
        })

    return results

In [ ]:
userQ = "I need buy device with 4500$"
retrieve_bm25(userQ, policy_texts, bm25, top_k=5)

[{'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium',
  'method': 'bm25'},
 {'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium',
  'method': 'bm25'},
 {'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium',
  'method': 'bm25'},
 {'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium',
  'method': 'bm25'},
 {'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true 

In [ ]:
def normalize_scores(results):

    scores = [r["Score"] for r in results]

    min_s = min(scores)
    max_s = max(scores)

    for r in results:

        if max_s == min_s:
            r["norm_score"] = 1.0
        else:
            r["norm_score"] = (r["Score"] - min_s) / (max_s - min_s)

    return results

In [ ]:
def hybrid_retrieve(userQ, policy_embeddings, policy_texts, bm25):

    semantic_results = retrive_cand(userQ, policy_embeddings)
    bm25_results = retrieve_bm25(userQ, policy_texts, bm25)

    semantic_results = normalize_scores(semantic_results)
    bm25_results = normalize_scores(bm25_results)

    combined = {}

    # semantic results
    for r in semantic_results:

        text = r["full_text"]

        combined[text] = {
            "full_text": text,
            "semantic": r["norm_score"],
            "bm25": 0
        }

    # bm25 results
    for r in bm25_results:

        text = r["full_text"]

        if text not in combined:

            combined[text] = {
                "full_text": text,
                "semantic": 0,
                "bm25": r["norm_score"]
            }

        else:

            combined[text]["bm25"] = r["norm_score"]

    final = []

    for item in combined.values():
## final_score = 0.7 * semantic_score + 0.3 * bm25_score

        score = (
            0.75 * item["semantic"] +
            0.25 * item["bm25"]
        )

        final.append({
            "Score": round(score,3),
            "full_text": item["full_text"]
        })

    final = sorted(final, key=lambda x: x["Score"], reverse=True)

    return final[:5]

In [ ]:
userQ = "I need buy device with 4000$"

hybrid_top = hybrid_retrieve(
    userQ,
    policy_embeddings,
    policy_texts,
    bm25
)

for r in hybrid_top:

    print("Score:", r["Score"])
    print("Text :", r["full_text"])
    print("-"*60)

Score: 1.0
Text : Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
------------------------------------------------------------
Score: 0.705
Text : Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
------------------------------------------------------------
Score: 0.673
Text : Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
------------------------------------------------------------
Score: 0.509
Text : Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
------------------------------------------------------------
Score: 0.25
Text : Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | n

# Tasks — Building a Practical RAG Decision System


In this lab you will extend the **Policy RAG system** step by step.

The system currently can:

* convert policies to embeddings
* retrieve **Top‑5 policies using semantic search**
* check **numeric rules**

Now you will extend the system to make it closer to a **real AI decision system**.

---

## Task 1 — Compare Retrieval Methods

First, test the difference between **Semantic Search** and **BM25 Search** VS ** Matching rule with condition col**.

Run the same user query using both retrieval methods.

Example query:

```
I need to buy a device for 3500$
```

Run:

* semantic retrieval
* Matching rule with condition col
* BM25 retrieval

### Experiment

Observe the results:

* Which policies appear in **semantic search**?
* Which policies appear in **Matching rule with condition col**?
* Which policies appear in **BM25**?

Try more queries:

```
buy device 3500
hardware purchase
vendor agreement
purchase approval
```

---

## Task 2 — Use the LLM to Re‑Rank Candidates

Now add the **LLM to the retrieval pipeline**.

Instead of trusting the retrieval scores directly, ask the LLM to choose the **best rule from the candidates**.

Input to the LLM:

* the user question
* the retrieved candidate policies

The LLM should return **the single most relevant rule**.

### Example

User question

```
I need to buy a device for 3500$
```

Candidate rules

```
amount <= 1500
1500 < amount <= 3000
amount > 3000
```

Expected LLM output

```
amount > 3000
```

### Experiment

Run the system with different queries and see how the LLM selects rules.

---

## Task 3 — Apply Rule Filtering

After the LLM selects a rule candidate, the system must verify that the rule **actually matches the user request**.

Example rule

```
amount > 3000
```

Example request

```
buy device for 3500$
```

The system should check if the condition is **true**.

### Experiment

Test the system with:

```
buy device for 800$
buy device for 1800$
buy device for 3500$
```

Observe how the selected rule changes.

---

## Task 4 — Let the LLM Explain the Decision

Now use the LLM to generate a **short explanation** for the decision.

Input:

* user question
* matched rule

Example

User request

```
buy device for 4000$
```

Rule

```
amount > 3000
```

Example explanation

```
The requested amount is higher than the policy limit of 3000$, therefore higher approval is required.
```

### Experiment

Try multiple queries and observe how the explanation changes.

---

## Task 5 — LLM Guardrail Task

Sometimes LLMs may generate answers that **do not follow the policy rules**.

In real systems we must prevent this.

Your task is to build a **Guardrail** that ensures the LLM only answers using valid policies.

### Goal

Create a validation step before returning the LLM answer.

The guardrail should check:

* Is the answer based on a retrieved policy?
* Does the rule condition match the user request?

If not, the system should **reject the response**.

### Example

User request

```
Can I buy a device for 10000$ without approval?
```

The LLM might try to answer freely.

The guardrail should force the system to respond using the correct policy.

Example safe response

```
According to the Hardware Purchase Policy, purchases above $3000 require manager, finance, and CTO approval.
```

### Experiment

Try questions that might confuse the model:

```
Can I skip approval for a 5000$ purchase?
Is there a way to bypass the purchase limit?
Can I buy hardware without following policy?
```

Check if your guardrail forces the system to follow the correct rule.

---
## Task 5 — Build the Final AI Decision System

Now combine everything into a complete pipeline.

Final pipeline:

```
User Question
↓
Hybrid Retrieval
↓
Candidate Policies
↓
LLM Re‑Ranking
↓
Rule Filtering
↓
LLM Explanation
```

Create a single function that runs the entire pipeline.

### Experiment

Run the system with different questions:

```
I need to buy device for 800$
I need to buy device for 1800$
I need to buy device for 3500$
I need a flight for 4 hours
I need a flight for 12 hours
```
Observe how the decision changes.

---


In [ ]:
!pip install bitsandbytes accelerate -q

In [ ]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_PATH = r"/content/drive/MyDrive/hf_models/Phi_3_5_mini_instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16,
    local_files_only=True
)

print("✅ Model loaded")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model loaded


In [ ]:
def ask_llm(messages, max_new_tokens=512):
    """Send messages to Phi-3.5 and return the response."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Take first answer only if duplicated
    if result.count("Best Match") > 1:
        parts = result.split("Best Match")
        result = "Best Match" + parts[1]

    return result

In [ ]:
def extract_number(text, keyword):
    """Extract numeric value from user query based on keyword type."""
    patterns = {
        'amount': [
            r'\$([\d,]+)', r'([\d,]+)\s*\$',
            r'([\d,]+)\s*dollar', r'cost.*?([\d,]+)',
            r'price.*?([\d,]+)', r'budget.*?([\d,]+)',
        ],
        'flight_hours': [
            r'(\d+)\s*hours?\s*flight', r'(\d+)\s*hr',
            r'flight.*?(\d+)\s*hours?', r'(\d+)\s*hours?',
        ]
    }
    for pattern in patterns.get(keyword, []):
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return int(match.group(1).replace(",", ""))
    return None


def safe_eval_condition(expr):
    """Safely evaluate a numeric condition (no code injection)."""
    cleaned = re.sub(r'\b(and|or|not|True|False|None)\b', '', expr)
    if not re.match(r'^[\d\s\.\+\-\*/<>=!()]+$', cleaned.strip()):
        return False
    try:
        return eval(expr, {"__builtins__": {}}, {})
    except:
        return False


def filter_by_condition(userQ, candidates):
    """Filter candidates by evaluating their numeric conditions."""
    amount_val = extract_number(userQ, 'amount')
    flight_hours_val = extract_number(userQ, 'flight_hours')

    matched = []
    for cand in candidates:
        condition = cand["condition"]
        expr = condition

        if amount_val is not None:
            expr = expr.replace("amount", str(amount_val))
        if flight_hours_val is not None:
            expr = expr.replace("flight_hours", str(flight_hours_val))

        # Fix SQL-style AND/OR → Python and/or
        expr = re.sub(r'\bAND\b', 'and', expr)
        expr = re.sub(r'\bOR\b', 'or', expr)

        # Handle mixed conditions (numeric + string like role==executive)
        if re.search(r'[a-zA-Z_]+\s*==\s*[a-zA-Z_]+', expr):
            parts = re.split(r'\b(?:and|or)\b', expr, flags=re.IGNORECASE)
            numeric_ok = []
            for part in parts:
                part = part.strip()
                if not re.search(r'[a-zA-Z_]', part) and part:
                    numeric_ok.append(safe_eval_condition(part))
            if numeric_ok and all(numeric_ok):
                matched.append({
                    **cand, "evaluated": expr,
                    "partial_match": True,
                    "note": "Numeric condition passed; non-numeric needs user context"
                })
            continue

        if safe_eval_condition(expr):
            matched.append({**cand, "evaluated": expr})

    return matched, amount_val, flight_hours_val

# Task 1

In [ ]:
def compare_retrieval(userQ, policy_embeddings, policy_texts, bm25):
    """Compare Semantic, Condition Matching, and BM25 for the same query."""

    print(f"\n{'='*70}")
    print(f"  Query: {userQ}")
    print(f"{'='*70}")

    # 1) Semantic Search
    sem = retrive_cand(userQ, policy_embeddings)
    print(f"\n📌 Semantic Search (Top 5):")
    for i, r in enumerate(sem, 1):
        print(f"  {i}. [Score={r['Score']}] {r['full_text'][:100]}...")

    # 2) Condition Matching
    matched, amt, hrs = filter_by_condition(userQ, sem)
    print(f"\n📌 Condition Matching (amount={amt}, flight_hours={hrs}):")
    if matched:
        for r in matched:
            tag = " [PARTIAL]" if r.get("partial_match") else ""
            print(f"  ✓{tag} {r['condition']} → {r.get('evaluated', 'N/A')}")
    else:
        print("  ✗ No numeric conditions matched")

    # 3) BM25
    bm = retrieve_bm25(userQ, policy_texts, bm25, top_k=5)
    print(f"\n📌 BM25 Search (Top 5):")
    for i, r in enumerate(bm, 1):
        print(f"  {i}. [Score={r['Score']}] {r['full_text'][:100]}...")

    return sem, matched, bm

In [ ]:
# Experiment: run multiple queries
queries = [
    "I need to buy a device for 3500$",
    "buy device 3500",
    "hardware purchase",
    "vendor agreement",
    "purchase approval",
]

for q in queries:
    compare_retrieval(q, policy_embeddings, policy_texts, bm25)


  Query: I need to buy a device for 3500$

📌 Semantic Search (Top 5):
  1. [Score=0.484] Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 ...
  2. [Score=0.425] Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= ...
  3. [Score=0.373] Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO appro...
  4. [Score=0.351] Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annua...
  5. [Score=0.256] Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | ven...

📌 Condition Matching (amount=3500, flight_hours=None):
  ✓ amount > 3000 → 3500 > 3000

📌 BM25 Search (Top 5):
  1. [Score=0.0] Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= ...
  2. [Score=0.0] Financial | Hardware Purchase Policy | Purchases between $1501–$

# Task 2

In [ ]:
def llm_rerank(userQ, candidates):
    """Ask the LLM to pick the single best matching policy rule."""
    candidates_text = ""
    for idx, cand in enumerate(candidates, 1):
        candidates_text += f"Candidate {idx} (score={cand['Score']}):\n"
        candidates_text += f"  {cand['full_text']}\n"
        if cand.get("note"):
            candidates_text += f"  Note: {cand['note']}\n"
        candidates_text += "\n"

    messages = [
        {
            "role": "system",
            "content": """You are a corporate policy matching assistant.
Select the ONE best matching policy rule for the user's request.

Instructions:
1. Pay close attention to NUMERICAL values (amounts, hours, etc.).
2. Match the number against the threshold in each candidate.
   - $4500 does NOT match "$1501-$3000", it matches "above $3000".
   - 8 hours does NOT match "under 6 hours", it matches "over 6 hours".
3. Return EXACTLY this format:

Best Match: Candidate <number>
Rule: <the best candidate's full_text>
Reason: <brief explanation>"""
        },
        {
            "role": "user",
            "content": f"""User asked: \"{userQ}\"

Candidate policy rules:

{candidates_text}

Which candidate is the best match?"""
        }
    ]
    return ask_llm(messages, max_new_tokens=512)

In [ ]:
test_queries = [
    "I need to buy a device for 3500$",
    "I need to buy a device for 800$",
    "I need to buy a device for 1800$",
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {q}")
    print(f"{'='*60}")

    top5 = retrive_cand(q, policy_embeddings)
    filtered, _, _ = filter_by_condition(q, top5)
    candidates = filtered if filtered else top5

    result = llm_rerank(q, candidates)
    print(result)


Query: I need to buy a device for 3500$
Best Match: Candidate 1
Rule: Purchases above $3000 require manager, finance, and CTO approval
Reason: The user's request for a device for $3500 exceeds the $3000 threshold mentioned in the policy, necessitating approval from manager, finance, and CTO as per the policy rule.

Query: I need to buy a device for 800$
Best Match: Candidate 1
Rule: Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved
Reason: The user's request to buy a device for $800 falls within the threshold of $1500 mentioned in Candidate 1's policy rule, which means the purchase would be auto-approved according to this policy.

Query: I need to buy a device for 1800$
Best Match: Candidate 1
Rule: Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes
Reason: The user's request for a device purchase at $1800 falls within the specified range of $1501-$3000, which according to the policy rule, requires ma

# Task 3

In [ ]:
def verify_rule(userQ, rule_condition):
    """Verify that a rule condition actually matches the user's request."""
    amount_val = extract_number(userQ, 'amount')
    flight_hours_val = extract_number(userQ, 'flight_hours')

    expr = rule_condition
    if amount_val is not None:
        expr = expr.replace("amount", str(amount_val))
    if flight_hours_val is not None:
        expr = expr.replace("flight_hours", str(flight_hours_val))

    expr = re.sub(r'\bAND\b', 'and', expr)
    expr = re.sub(r'\bOR\b', 'or', expr)

    # If still has unresolved variables, check numeric parts only
    if re.search(r'[a-zA-Z_]', expr):
        parts = re.split(r'\b(?:and|or)\b', expr, flags=re.IGNORECASE)
        numeric_results = []
        for part in parts:
            part = part.strip()
            if not re.search(r'[a-zA-Z_]', part) and part:
                numeric_results.append(safe_eval_condition(part))
        if numeric_results:
            passed = all(numeric_results)
            if passed:
                return True, expr, "partial (numeric passed, non-numeric needs user context)"
            else:
                return False, expr, "failed (numeric condition is False)"
        return False, expr, "cannot evaluate"

    result = safe_eval_condition(expr)
    return result, expr, "full match" if result else "condition failed"


In [ ]:
test_cases = [
    # ── Amount tests ──
    ("buy device for 800$",  "amount <= 1500"),
    ("buy device for 800$",  "1500 < amount <= 3000"),
    ("buy device for 800$",  "amount > 3000"),
    ("buy device for 1800$", "1500 < amount <= 3000"),
    ("buy device for 3500$", "amount > 3000"),
    ("buy device for 3500$", "amount <= 1500"),

    # ── Flight hours tests ──
    ("I need a flight for 4 hours",  "flight_hours <= 6"),
    ("I need a flight for 4 hours",  "flight_hours > 6 AND role == executive"),
    ("I need a flight for 8 hours",  "flight_hours <= 6"),
    ("I need a flight for 8 hours",  "flight_hours > 6 AND role == executive"),
    ("I need a flight for 12 hours", "flight_hours > 6 AND role == executive"),
    ("I need a flight for 12 hours", "flight_hours <= 6"),
]
rows = []
for query, condition in test_cases:
    passed, evaluated, status = verify_rule(query, condition)

    if passed and "numeric passed" in status.lower():
        result = "⚠️ Partial"
    elif passed:
        result = "✅ Pass"
    else:
        result = "❌ Fail"

    rows.append({
        "Query": query,
        "Condition": condition,
        "Evaluated": evaluated,
        "Result": result,
        "Status": status
    })

df_results = pd.DataFrame(rows)

def highlight_result(val):
    if "Pass" in val:
        return "color: green; font-weight: bold"
    elif "Partial" in val:
        return "color: orange; font-weight: bold"
    else:
        return "color: red; font-weight: bold"

styled = (
    df_results
    .style
    .map(highlight_result, subset=["Result"])
    .set_properties(**{
        "text-align": "left",
        "font-size": "13px"
    })
)

styled

,Query,Condition,Evaluated,Result,Status
0,buy device for 800$,amount <= 1500,800 <= 1500,✅ Pass,full match
1,buy device for 800$,1500 < amount <= 3000,1500 < 800 <= 3000,❌ Fail,condition failed
2,buy device for 800$,amount > 3000,800 > 3000,❌ Fail,condition failed
3,buy device for 1800$,1500 < amount <= 3000,1500 < 1800 <= 3000,✅ Pass,full match
4,buy device for 3500$,amount > 3000,3500 > 3000,✅ Pass,full match
5,buy device for 3500$,amount <= 1500,3500 <= 1500,❌ Fail,condition failed
6,I need a flight for 4 hours,flight_hours <= 6,4 <= 6,✅ Pass,full match
7,I need a flight for 4 hours,flight_hours > 6 AND role == executive,4 > 6 and role == executive,❌ Fail,failed (numeric condition is False)
8,I need a flight for 8 hours,flight_hours <= 6,8 <= 6,❌ Fail,condition failed
9,I need a flight for 8 hours,flight_hours > 6 AND role == executive,8 > 6 and role == executive,⚠️ Partial,"partial (numeric passed, non-numeric needs user context)"


# Task 4

In [ ]:
def llm_explain(userQ, matched_rule, rule_condition, verified):
    """Ask the LLM to explain the policy decision to the user."""
    status = "MATCHES" if verified else "DOES NOT MATCH"

    messages = [
        {
            "role": "system",
            "content": """You are a corporate policy assistant.
Explain the policy decision clearly in 2-3 sentences.
Be specific about the numbers (amount, hours) and what approval is needed.
Do NOT invent any policy — only use the rule provided."""
        },
        {
            "role": "user",
            "content": f"""User request: "{userQ}"

Matched policy rule: {matched_rule}
Rule condition: {rule_condition}
Verification: The condition {status} the user's request.

Explain this decision to the user clearly."""
        }
    ]
    return ask_llm(messages, max_new_tokens=256)

In [ ]:
test_queries_t4 = [
    "I need to buy a device for 800$",
    "I need to buy a device for 1800$",
    "I need to buy a device for 4000$",
]

for q in test_queries_t4:
    print(f"\n{'='*60}")
    print(f"Query: {q}")

    top5 = retrive_cand(q, policy_embeddings)
    filtered, _, _ = filter_by_condition(q, top5)
    candidates = filtered if filtered else top5
    best_rule = candidates[0]["full_text"] if candidates else "No rule found"
    best_cond = candidates[0]["condition"] if candidates else ""
    verified = bool(filtered)

    explanation = llm_explain(q, best_rule, best_cond, verified)
    print(f"\n💡 Explanation:\n{explanation}")


Query: I need to buy a device for 800$

💡 Explanation:
According to our Hardware Purchase Policy, any hardware purchase request under $1500 is automatically approved without the need for additional approvals. Since the device you wish to buy costs $800, which is below the $1500 threshold, your purchase qualifies for auto-approval. Therefore, you can proceed with the purchase without any further authorization required.

Query: I need to buy a device for 1800$

💡 Explanation:
According to our Hardware Purchase Policy, any hardware purchase amounting between $1,500 and $3,000 necessitates managerial approval. Since your intended purchase is $1,800, it falls within this specified range. Therefore, you will need to obtain approval from a manager to proceed with the purchase. Additionally, you are required to acquire at least two quotes for the device to ensure competitive pricing and to make an informed decision. This process is in place to safeguard the company's financial interests and t

# Task 5

In [ ]:
def guardrail_check(userQ, llm_response, matched_rules):
    """
    LLM-based Guardrail (Zero-Shot): Detects policy bypass attempts
    by analyzing USER QUERY only.
    """
    if not matched_rules:
        return False, "⚠️ No matching policy found. Please contact your manager."

    best_rule = matched_rules[0]

    guard_messages = [
        {
            "role": "system",
            "content": """You are a corporate policy compliance guard.

Analyze the user's query and determine their intent.

There are only two types of queries:

1. SAFE — The user is making a normal request or asking what the policy says.
   They accept that rules exist and want to know how they apply to their situation.

2. BLOCKED — The user is trying to work around the policy in ANY way, including:
   - Skipping or avoiding the required process entirely
   - Reducing the number of required approvers (e.g. wanting only one approver when the policy requires multiple)
   - Questioning whether the rules are necessary or negotiable
   - Suggesting that the limits or thresholds could be bent or changed
   - Implying they want to act without fully following the stated process

The key distinction: a SAFE user accepts the policy as it is and works WITHIN it.
A BLOCKED user wants to change, reduce, simplify, or ignore any part of it.

Respond with exactly one word: SAFE or BLOCKED"""
        },
        {
            "role": "user",
            "content": f"User query: \"{userQ}\"\n\nSAFE or BLOCKED?"
        }
    ]

    guard_result = ask_llm(guard_messages, max_new_tokens=5).strip()
    guard_clean = guard_result.upper().split()[0] if guard_result.strip() else "SAFE"

    print(f"    Guard LLM verdict: {guard_clean}")

    if "BLOCK" in guard_clean:
        safe = (
            f"❌ Request denied. According to company policy, "
            f"your request falls under: {best_rule['full_text'][:200]}. "
            f"The required approval process must be followed — it cannot be skipped or bypassed."
        )
        return False, safe

    return True, llm_response


In [ ]:
# Test Guardrail with tricky queries
# Includes queries that keyword detection would MISS!
tricky_queries = [
    # Obvious bypass attempts (keywords could catch these)
    "Can I buy a device for 10000$ without approval?",
    "Can I skip approval for a 5000$ purchase?",
    "Is there a way to bypass the purchase limit?",
    "Can I buy hardware without following policy?",

    # Subtle bypass attempts (ONLY LLM can catch these!)
    "Do I really need all those approvals for a 5000$ device?",
    "What if I just go ahead and buy it for 8000$?",
    "Is the 3000$ limit flexible?",
    "Can my manager handle the 6000$ purchase alone?",
]

for q in tricky_queries:
    print(f"\n{'='*60}")
    print(f"  Query: {q}")
    print(f"{'='*60}")

    top5 = retrive_cand(q, policy_embeddings)
    filtered, _, _ = filter_by_condition(q, top5)
    candidates = filtered if filtered else top5

    # Get LLM response
    llm_response = llm_rerank(q, candidates)

    # Apply Guard LLM
    is_safe, final_response = guardrail_check(q, llm_response, candidates)

    if is_safe:
        print(f"  ✅ Safe → {final_response[:200]}")
    else:
        print(f"  🛡️ BLOCKED → {final_response[:200]}")



  Query: Can I buy a device for 10000$ without approval?
    Guard LLM verdict: BLOCKED
  🛡️ BLOCKED → ❌ Request denied. According to company policy, your request falls under: Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 300

  Query: Can I skip approval for a 5000$ purchase?
    Guard LLM verdict: BLOCKED
  🛡️ BLOCKED → ❌ Request denied. According to company policy, your request falls under: Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 300

  Query: Is there a way to bypass the purchase limit?
    Guard LLM verdict: BLOCKED
  🛡️ BLOCKED → ❌ Request denied. According to company policy, your request falls under: Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium. The req

  Query: Can I buy hardware without following policy?
    Guard LLM verdict: BLOCKED
  🛡️ BLO

# Task 6

In [ ]:
def ai_decision_system(userQ, policy_embeddings, policy_texts, bm25):
    """
    Complete AI Decision Pipeline:
    1. Hybrid Retrieval (Semantic + BM25)
    2. Candidate Policies
    3. LLM Re-Ranking
    4. Rule Filtering
    5. LLM Explanation
    6. Guardrail Check
    """
    print(f"\n{'='*70}")
    print(f"  🤖 AI Decision System")
    print(f"  Query: {userQ}")
    print(f"{'='*70}")

    # Step 1: Hybrid Retrieval
    hybrid_results = hybrid_retrieve(userQ, policy_embeddings, policy_texts, bm25)
    print(f"\n📌 Step 1 — Hybrid Retrieval: {len(hybrid_results)} candidates")
    for i, r in enumerate(hybrid_results, 1):
        print(f"  {i}. [Score={r['Score']}] {r['full_text'][:90]}...")

    # Add condition info to hybrid results
    for r in hybrid_results:
        # Find condition from df
        for idx, row in df.iterrows():
            if row["full_text"] == r["full_text"]:
                r["condition"] = str(row["condition"])
                r["df_idx"] = idx
                break
        if "condition" not in r:
            r["condition"] = "nan"

    # Step 2: Rule Filtering
    filtered, amount_val, flight_hours_val = filter_by_condition(userQ, hybrid_results)
    print(f"\n📌 Step 2 — Rule Filtering (amount={amount_val}, hours={flight_hours_val}):")
    print(f"  → {len(filtered)} rules passed")
    for c in filtered:
        tag = " [PARTIAL]" if c.get("partial_match") else ""
        print(f"    ✓{tag} {c['condition']} → {c.get('evaluated','')}")

    # Step 3: LLM Re-Ranking
    candidates_for_llm = filtered if filtered else hybrid_results
    print(f"\n📌 Step 3 — LLM Re-Ranking ({len(candidates_for_llm)} candidates):")
    llm_result = llm_rerank(userQ, candidates_for_llm)
    print(f"  {llm_result[:200]}...")

    # Step 4: LLM Explanation
    best = candidates_for_llm[0]
    verified = bool(filtered)
    print(f"\n📌 Step 4 — LLM Explanation:")
    explanation = llm_explain(userQ, best["full_text"], best["condition"], verified)
    print(f"  {explanation}")

    # Step 5: Guardrail
    is_safe, final = guardrail_check(userQ, explanation, candidates_for_llm)
    print(f"\n📌 Step 5 — Guardrail: {'✅ Safe' if is_safe else '🛡️ Overridden'}")

    # Final Answer
    print(f"\n{'─'*70}")
    print(f"  📋 Final Answer:")
    print(f"  {final}")
    print(f"{'─'*70}")

    return final

In [ ]:
# Run the complete system with all test queries
test_queries_final = [
    "I need to buy device for 800$",
    "I need to buy device for 1800$",
    "I need to buy device for 3500$",
    "I need a flight for 4 hours",
    "I need a flight for 12 hours",
]

for q in test_queries_final:
    ai_decision_system(q, policy_embeddings, policy_texts, bm25)


  🤖 AI Decision System
  Query: I need to buy device for 800$

📌 Step 1 — Hybrid Retrieval: 5 candidates
  1. [Score=1.0] Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager appro...
  2. [Score=0.821] Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | ...
  3. [Score=0.649] Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and...
  4. [Score=0.429] Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500...
  5. [Score=0.25] Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list |...

📌 Step 2 — Rule Filtering (amount=800, hours=None):
  → 1 rules passed
    ✓ amount <= 1500 → 800 <= 1500

📌 Step 3 — LLM Re-Ranking (1 candidates):
  Best Match: Candidate 1
Rule: Hardware Purchase Policy for purchases up to $1500 are auto-approved
Reason: The user's request for a device for $800 falls within the auto-approved r